# Astra AI - Production Backend (Google Colab)
**SIH26167 | ISRO SAC | Multimodal Remote Sensing Platform**

**Before running:** Go to **Runtime -> Change runtime type -> T4 GPU -> Save**

Then press **Ctrl+F9** (Run All) — everything is fully automated!

In [ ]:
# @title Step 1: Install Dependencies
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - change runtime!')
print('Installing Python dependencies...')
!pip install -q fastapi 'uvicorn[standard]' python-multipart sse-starlette pydantic pydantic-settings google-generativeai rasterio pyproj shapely timm einops python-magic aiofiles loguru
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('All dependencies ready!')


In [ ]:
# @title Step 2: Mount Google Drive & Load Models
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive

print('Mounting Google Drive...')
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
BACKEND = Path('/content/astra-backend')
BACKEND.mkdir(exist_ok=True)

# --- Load backend code ---
code_zip = DRIVE / 'astra_code_only.zip'
if not (BACKEND / 'main.py').exists():
    if code_zip.exists():
        print('Extracting backend code from Drive...')
        with zipfile.ZipFile(str(code_zip), 'r') as z:
            z.extractall(str(BACKEND))
        print('Code extracted!')
    else:
        print('astra_code_only.zip not found in Drive root.')
        print('Upload astra_code_only.zip (zip of the backend folder) to the root of your Google Drive.')
else:
    print('Backend code already present.')

# --- Copy model weights from Drive ---
models_dir = BACKEND / 'models'
ben_v2_dir = models_dir / 'bigearthnet_resnet50_v2_full'
change_dir = models_dir / 'change_detection' / 'checkpoints'
ben_v2_dir.mkdir(parents=True, exist_ok=True)
change_dir.mkdir(parents=True, exist_ok=True)
(BACKEND / 'uploads').mkdir(exist_ok=True)
(BACKEND / 'outputs').mkdir(exist_ok=True)

model_map = [
    (DRIVE / 'bigearthnet_resnet50_best.pth', ben_v2_dir / 'best_model.pth'),
    (DRIVE / 'change_detection_best.pth',     change_dir / 'best_model.pth'),
    (DRIVE / 'ben_classifier.pth',             models_dir / 'ben_classifier.pth'),
]

for src, dst in model_map:
    if dst.exists():
        print(f'  Already copied: {dst.name}')
    elif src.exists():
        print(f'  Copying {src.name} -> {dst} ...')
        shutil.copy2(str(src), str(dst))
        print(f'  Done!')
    else:
        print(f'  WARNING: {src.name} not found in Drive root!')

print('\nAll models ready!')


In [ ]:
# @title Step 3: Start Backend + Launch Cloudflare Tunnel
import os, sys, time, re, subprocess, urllib.request, json
from pathlib import Path

# --- Gemini API Key ---
gemini_key = ''
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY') or ''
except Exception:
    pass
if not gemini_key:
    gemini_key = input('Paste your GEMINI_API_KEY (get one free at aistudio.google.com): ').strip()

os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['HOST'] = '0.0.0.0'
os.environ['PORT'] = '8000'
os.environ['ALLOWED_ORIGINS'] = '*'

!pkill -f 'uvicorn main:app' 2>/dev/null; pkill -f 'cloudflared' 2>/dev/null; sleep 1

print('Starting Astra FastAPI server...')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/astra-backend',
    stdout=open('/tmp/uvicorn.log', 'w'),
    stderr=subprocess.STDOUT
)

ready = False
for i in range(30):
    time.sleep(1)
    try:
        with urllib.request.urlopen('http://localhost:8000/health', timeout=2) as r:
            data = json.loads(r.read())
            if data.get('status') == 'online':
                gpu = data.get('gpu', {})
                print(f'Backend online! GPU: {gpu.get("name","CPU")} | VRAM free: {gpu.get("vram_free_gb")} GB')
                ready = True
                break
    except Exception:
        print(f'  Waiting... ({i+1}/30)')

if not ready:
    print('Backend failed. Logs:')
    !cat /tmp/uvicorn.log
    raise SystemExit()

print('Starting Cloudflare tunnel...')
cf = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

tunnel_url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = cf.stderr.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if tunnel_url:
    pair = f'https://sih-orbilens-2026-tm.web.app/backend-config.html?url={tunnel_url}'
    print('\n' + '='*65)
    print('ASTRA AI IS LIVE!')
    print('='*65)
    print(f'Tunnel URL  : {tunnel_url}')
    print(f'API Docs    : {tunnel_url}/docs')
    print('='*65)
    print('\n1-CLICK FIREBASE PAIRING:')
    print(pair)
    print('\nKeep this tab open while using the app!')
else:
    print('Tunnel not found - check cloudflared.')
